<a href="https://colab.research.google.com/github/andiunsia/Latihan_Data_Science/blob/main/Pertemuan12_Andi_240401010009.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
Nama : Andi

NIM : 240401010009

Kelas : IF401

---

## Generate & Eksplorasi Dataset Transaksi

In [15]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur', 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Generate 50 transaksi sintetis dengan jumlah item 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    items = np.random.choice(produk, n_item, replace=False).tolist()
    transaksi.append(items)

import warnings
warnings.filterwarnings("ignore")

# Menambahkan pola pembelian bersama antara Roti dan Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

# ONE-HOT ENCOSING TRANSAKSI
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df.head())


Contoh transaksi: [['Keju', 'Roti', 'Mentega', 'Kopi', 'Selai'], ['Roti', 'Kopi', 'Teh', 'Selai', 'Mentega'], ['Kopi', 'Susu', 'Teh']]
Jumlah transaksi: 50
    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


## Cari Frequent Itemset dengan Apriori

In [16]:
# Hitung frekuensi kemunculan tiap produk
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

print('Eksperimen Nilai min_support :')
for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

import warnings
warnings.filterwarnings("ignore")

# Memilih nilai minimum support yang menghasilkan frequent itemset dengan jumlah yang representatif
# Menggunakan min_support = 0.1 untuk analisis selanjutnya
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)

print('\n 10 Frequent Itemset Teratas :')
print(freq_items.head(10).to_string(index=False))


Eksperimen Nilai min_support :
min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan

 10 Frequent Itemset Teratas :
 support     itemsets
    0.52      (Selai)
    0.46        (Teh)
    0.42    (Mentega)
    0.36      (Telur)
    0.34       (Keju)
    0.32       (Gula)
    0.32       (Kopi)
    0.32       (Roti)
    0.32       (Susu)
    0.24 (Teh, Selai)


In [19]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence',
                           min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

import warnings
warnings.filterwarnings("ignore")

print('\n 10 Aturan Asosiasi Terkuat')
print(rules[['antecedents', 'consequents',
             'support', 'confidence', 'lift']].head(10))


 10 Aturan Asosiasi Terkuat
         antecedents consequents  support  confidence      lift
8        (Keju, Teh)     (Telur)     0.12    0.857143  2.380952
15  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
11      (Roti, Gula)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
10      (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
14     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
9      (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
12     (Selai, Gula)      (Roti)     0.10    0.500000  1.562500
13   (Kopi, Mentega)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


## **Aturan yang paling kuat, dengan Lift tertinggi :**
Kombinasi aturan ({Teh, Keju}) mencetak nilai Lift tertinggi sebesar 2.38

## **Apakah masuk akal secara bisnis (mis. Roti -> Selai)?**
Ya, pola ini masuk akal secara bisnis. Dengan nilai Lift >1 menunjukkan korelasi yang positif.
Kombinasi Roti -> Selai memiliki nilai Lift = 1.322115 berarti peluang pelanggan membeli Selai meningkat 32.2% jika sudah membeli Roti.
Karena angka Lift di atas 1, aturan ini valid dan secara statistik layak digunakan untuk strategi bisnis (seperti bundling atau rekomendasi).


## Rekomender Sederhana dengan Content-Based Filtering

In [25]:
from sklearn.metrics.pairwise import cosine_similarity
katalog = pd.DataFrame({
 'produk': produk,
 'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy', 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
 idx = katalog.index[katalog['produk'] == nama_produk][0]
 skor = list(enumerate(sim_matrix[idx]))
 skor = sorted(skor, key=lambda x: x[1], reverse=True)
 skor = [s for s in skor if s[0] != idx][:top_n]
 return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()


import warnings
warnings.filterwarnings("ignore")

# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung produk_target
rules_terkait = rules[rules['antecedents'].apply( lambda x: produk_target in x)]
print('REKOMENDASI PRODUK :')
print('1. Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('')
print('2. Rekomendasi via Content-Based Filtering :')
print(rekomendasi_serupa(produk_target))

REKOMENDASI PRODUK :
1. Rekomendasi dari Association Rules:
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115

2. Rekomendasi via Content-Based Filtering :
['Selai', 'Sereal', 'Susu']


## **Perbandingan Pendekatan :**

1. **Konsistensi Rekomendasi :** Kedua pendekatan baik *Association Rules* maupun *Content-Based Filtering* menunjukkan hasil yang konsisten. Untuk target produk Roti, kedua pendekatan sama-sama merekomendasikan Selai karena aturan terkuat, tetapi rekomendasi *Content-Based Filtering* lebih banyak yaitu Selai, Sereal, dan Susu karena berada pada kategori yang sama.

2. **Penggunaan Pendekatan :**
   * Association Rules : Cocok untuk promosi cross-selling, misalnya paket bundling produk yang saling melengkapi.
   * Content-Based Filtering : Lebih optimal untuk menangani produk baru yang belum punya riwayat transaksi di kasir, cukup merekomendasikannya berdasarkan kesamaan kategori produk.
   * Hybrid : Penggabungan kedua metode saling menutupi kelemahan, menggunakan Content-Based Filtering untuk menampilkan "Produk Serupa" di bagian bawah etalase produk, dan menyodorkan rekomendasi dari Association Rules ketika barang sudah berada di keranjang belanja menjelang checkout.


